In [78]:
from databricks.connect import DatabricksSession
spark = DatabricksSession.builder.getOrCreate()

from pyspark.sql import Window
from pyspark.sql.functions import date_format,sum, lag, col, round, year, month

In [ ]:
# variables
# interactive widgets for catalog and schema names
dbutils.widgets.text("catalog", "nyctaxi_project_dev")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema", "gold")

catalog = dbutils.widgets.get("catalog")
source_schema = dbutils.widgets.get("silver_schema")
target_schema = dbutils.widgets.get("gold_schema")

In [ ]:
gold_df = spark.table(f"{catalog}.{source_schema}.yellow_trips_enriched")
# display(gold_df)

,trip_id,vendor,pickup_datetime,dropoff_datetime,pickup_borough,pickup_zone,pickup_service_zone,dropoff_borough,dropoff_zone,dropoff_service_zone,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,payment_method,rate_name,congestion_surcharge,airport_fee,cbd_congestion_fee,ingestion_timestamp,source_file_name,source_file_path
0,-9223348661467009949,"Creative Mobile Technologies, LLC",2025-07-18 11:39:03,2025-07-18 12:11:01,Manhattan,Upper East Side South,Yellow Zone,Manhattan,Hudson Sq,Yellow Zone,1.0,4.10,26.80,3.25,0.5,9.45,0.00,1.0,41.00,Credit Card,Standard rate,2.5,0.0,0.75,2026-09-05 13:08:34.712730,yellow_tripdata_2025-07.parquet,dbfs:/Volumes/nyctaxi_project/landing/yellow_taxi/raw_trips/yellow_tripdata_2025-07.parquet
1,-9223267054186452619,"Curb Mobility, LLC",2025-10-23 16:25:20,2025-10-23 16:40:36,Manhattan,Upper East Side North,Yellow Zone,Manhattan,Manhattan Valley,Yellow Zone,1.0,2.02,15.60,2.50,0.5,1.77,0.00,1.0,23.87,Credit Card,Standard rate,2.5,0.0,0.00,2026-09-05 13:08:34.712730,yellow_tripdata_2025-10.parquet,dbfs:/Volumes/nyctaxi_project/landing/yellow_taxi/raw_trips/yellow_tripdata_2025-10.parquet
2,-9223236955286100166,"Curb Mobility, LLC",2026-01-08 18:40:04,2026-01-08 18:54:04,Manhattan,Upper East Side South,Yellow Zone,Manhattan,Midtown East,Yellow Zone,1.0,1.45,13.50,2.50,0.5,4.15,0.00,1.0,24.90,Credit Card,Standard rate,2.5,0.0,0.75,2026-09-05 13:08:34.712730,yellow_tripdata_2026-01.parquet,dbfs:/Volumes/nyctaxi_project/landing/yellow_taxi/raw_trips/yellow_tripdata_2026-01.parquet
3,-9223152677722452442,"Curb Mobility, LLC",2025-06-08 08:52:29,2025-06-08 09:04:54,Manhattan,Times Sq/Theatre District,Yellow Zone,Manhattan,Sutton Place/Turtle Bay North,Yellow Zone,1.0,1.49,12.80,0.00,0.5,0.00,0.00,1.0,17.55,Credit Card,Standard rate,2.5,0.0,0.75,2026-09-05 13:08:34.712730,yellow_tripdata_2025-06.parquet,dbfs:/Volumes/nyctaxi_project/landing/yellow_taxi/raw_trips/yellow_tripdata_2025-06.parquet
4,-9223108776888658715,"Curb Mobility, LLC",2025-05-21 07:17:37,2025-05-21 07:21:33,Manhattan,Flatiron,Yellow Zone,Manhattan,Times Sq/Theatre District,Yellow Zone,1.0,0.99,7.20,0.00,0.5,2.39,0.00,1.0,14.34,Credit Card,Standard rate,2.5,0.0,0.75,2026-09-05 13:08:34.712730,yellow_tripdata_2025-05.parquet,dbfs:/Volumes/nyctaxi_project/landing/yellow_taxi/raw_trips/yellow_tripdata_2025-05.parquet
5,-9223082165135881414,"Curb Mobility, LLC",2025-11-12 11:53:54,2025-11-12 12:02:32,Manhattan,Gramercy,Yellow Zone,Manhattan,Union Sq,Yellow Zone,1.0,0.53,9.30,0.00,0.5,0.00,0.00,1.0,14.05,Cash,Standard rate,2.5,0.0,0.75,2026-09-05 13:08:34.712730,yellow_tripdata_2025-11.parquet,dbfs:/Volumes/nyctaxi_project/landing/yellow_taxi/raw_trips/yellow_tripdata_2025-11.parquet
6,-9223072419320970721,"Curb Mobility, LLC",2025-11-02 18:56:33,2025-11-02 19:02:53,Queens,Old Astoria,Boro Zone,Queens,Old Astoria,Boro Zone,NaN,1.04,-1.50,0.00,0.5,0.00,0.00,1.0,5.00,Flex Fare trip,Null/unknown,0.0,0.0,0.00,2026-09-05 13:08:34.712730,yellow_tripdata_2025-11.parquet,dbfs:/Volumes/nyctaxi_project/landing/yellow_taxi/raw_trips/yellow_tripdata_2025-11.parquet
7,-9223054119336668189,"Curb Mobility, LLC",2025-09-22 16:12:08,2025-09-22 16:26:56,Manhattan,Upper East Side South,Yellow Zone,Manhattan,Upper West Side South,Yellow Zone,1.0,1.95,14.90,2.50,0.5,4.28,0.00,1.0,25.68,Credit Card,Standard rate,2.5,0.0,0.00,2026-09-05 13:08:34.712730,yellow_tripdata_2025-09.parquet,dbfs:/Volumes/nyctaxi_project/landing/yellow_taxi/raw_trips/yellow_tripdata_2025-09.parquet
8,-9223038480215781674,"Curb Mobility, LLC",2025-10-13 14:17:57,2025-10-13 15:03:56,Manhattan,Financial District North,Yellow Zone,Queens,JFK Airport,Airports,2.0,19.86,70.00,0.00,0.5,16.34,6.94,1.0,98.03,Credit Card,JFK,2.5,0.0,0.75,2026-09-05 13:08:34.712730,yellow_tripdata_2025-10.parquet,dbfs:/Volumes/nyctaxi_project/landing/yellow_taxi/raw_trips/yellow_tripdata_2025-10.parquet
9,-9223012440418345808,"Creative Mobile Technol

In [ ]:
# calculate 3-month rolling revenue per vendor 

golf_df = spark.table(f"{catalog}.{source_schema}.yellow_trips_enriched")
vendor_month_revenue = (
    gold_df
        .withColumn("year_month", date_format("pickup_datetime", "yyyy-MM"))
        .groupBy("vendor", "year_month")
        .sum("total_amount")
        .withColumnRenamed("sum(total_amount)", "total_revenue")

)

# display(vendor_month_revenue)

# find value for rolling 3 months
w = Window.partitionBy("vendor").orderBy("year_month").rowsBetween(-2, Window.currentRow)
w_lag = Window.partitionBy("vendor").orderBy("year_month")

vendor_revenue_3m = (
    vendor_month_revenue
        .withColumn("3Mrevenue", sum("total_revenue").over(w))
        .withColumn("prev", lag("3Mrevenue", 1).over(w_lag))
        .withColumn("vs_last_month", round(100 *(col("3Mrevenue") - col("prev")) / col("prev"), 2))
        .drop("total_revenue")
)

# display(vendor_revenue_3m)

vendor_revenue_3m.write.mode("overwrite").saveAsTable(f"{catalog}.{target_schema}.vendor_3m_revenue")

In [ ]:
# calculate YTD revenue per vendor, compare it against the last year YTD revenue and show diff in %

from pyspark.sql.functions import lit, current_date, date_trunc, add_months, to_date, nullif

gold_df = spark.table(f"{catalog}.{source_schema}.yellow_trips_enriched")


# define a date range for YTD -> it must be between Jan 1st, current_year and current_date

ytd_start = date_trunc("year", current_date())
ytd_end = current_date()

# define a date range for YTD-1 -> it must be between last year Jan 1st and current_date -1 year

ytd_ly_start = add_months(ytd_start, -12)
ytd_ly_end = add_months(ytd_end, -12)

# filter df for only YTD values

ytd = (
    gold_df
        .filter((to_date(col("pickup_datetime")) >= ytd_start) & (to_date(col("pickup_datetime")) <= ytd_end)
                   )
        .groupBy("vendor")
        .agg(
            sum("total_amount").alias("ytd_sales")
        )
)

ly_ytd = (
    gold_df
        .filter((to_date(col("pickup_datetime")) >= ytd_ly_start) & (to_date(col("pickup_datetime")) <= ytd_ly_end)
                   )
        .groupBy("vendor")
        .agg(
            sum("total_amount").alias("ly_ytd_sales")
        )
)


final = (
    ytd.join(ly_ytd, on='vendor', how='outer')
        .fillna(0, subset=["ytd_sales", "ly_ytd_sales"])
        .withColumn("vs_ly_pct", round(100*(col("ytd_sales") - col("ly_ytd_sales")) / nullif(col("ly_ytd_sales"), lit(0)), 2)
                    )
)

final.write.mode("overwrite").saveAsTable(f"{catalog}.{target_schema}.ytd_vs_ly_ytd_revenue_vendor")

# display(final)

,vendor,ytd_sales,ly_ytd_sales,vs_ly_pct
0,"Curb Mobility, LLC",2.624588e+08,4.898217e+08,-46.42
1,"Creative Mobile Technologies, LLC",6.012623e+07,1.233879e+08,-51.27
2,Myle Technologies Inc,5.984320e+05,2.762676e+05,116.61
3,Helix,3.452417e+06,7.899438e+06,-56.30


In [ ]:
# calculate total number of trips per month TO JFK Airport (dropoff_zone == 'JFK Airport') and FROM JFK Airport (pickup_zone == 'JFK Airport') as two separate series

# add period column to the dataset
# group by period and aggregate to_JFK trips 
# group by period and aggregate from_JFK trips

from pyspark.sql.functions import when

transform_df = (
    gold_df
        .withColumn("year_month", date_format("pickup_datetime", "yyyy-MM"))
        .withColumn("to_jfk", when(col("dropoff_zone") == "JFK Airport", lit(1)).otherwise(lit(0)))
        .withColumn("from_jfk", when(col("pickup_zone") == "JFK Airport", lit(1)).otherwise(lit(0)))
        )

jfk_trips = (
    transform_df
        .groupBy("year_month")
        .agg(
            sum("to_jfk").alias("total_trips_to_airport"),
            sum("from_jfk").alias("total_trips_from_airport")
        )
        .orderBy("year_month")
)

jfk_trips.write.mode("overwrite").saveAsTable(f"{catalog}.{target_schema}.monthly_trips_jfk")

In [ ]:
# calculate the most popular pickup and drop-off zones citywide (total number of trips), plus top zones ranked within each borough using window functions (rank/dense_rank)

In [ ]:
# calculate the most common pickup -> dropoff zone pairs (top 20 routes), not just zones independently
from pyspark.sql.functions import count, desc
top_routes = (
    gold_df
        .groupBy("pickup_zone", "dropoff_zone")
        .agg(
            count("trip_id").alias("total_trips")
        )
        .orderBy(desc("total_trips"))
        .limit(20)

)

top_routes.write.mode("overwrite").saveAsTable(f"{catalog}.{target_schema}.top_twenty_routes")

In [ ]:
# calculate revenue per passenger by month: sum(total_amount) / sum(passenger_count) per month, not a row-wise average of per-trip ratios



In [ ]:
# calculate the percentage of trips attributable to morning commute hours (7-9am), restricted to weekdays (Mon-Fri); compare against weekend mornings for contrast

from pyspark.sql.functions import dayofweek, hour, when, countDistinct, first
from pyspark.sql import Window

# add a column with an hour for each trip and add q column with a weekday

enriched_df = (
    gold_df
        .withColumns(
            {
                "hour": hour("pickup_datetime"),
                "dayofweek": dayofweek("pickup_datetime"),
                "date": to_date("pickup_datetime")
            }
        )
        .withColumn(
            "trip_type", 
                when((col("hour").isin([7,8,9])) & (col("dayofweek").isin([2,3,4,5,6])), "working_day_morning")
                .when(((col("hour").isin([7,8,9]))) & ((col("dayofweek").isin([1,7]))), "weekend_morning")
                .otherwise(lit("exclude"))
        )
        .filter((col("trip_type") == "working_day_morning") | (col("trip_type") == "weekend_morning"))
)

# calculate trips distribution and run rate 

trips_distribution = (
    enriched_df
        .groupBy("trip_type")
        .agg(
            count("*").alias("number_of_trips"),
            countDistinct("date").alias("total_days")

        )
        #.withColumn("pct_total_trips")
        .withColumn("run_rate_trips_per_day", round(col("number_of_trips") / col("total_days"),0))
)

w = Window.partitionBy()

final= (
    trips_distribution
        .withColumn("pct_total", round(100*(col("number_of_trips") / sum("number_of_trips").over(w)), 2))
)

display(final)


/Users/alehnaumovich/podyplomowe_genai_public/.venv/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


,trip_type,number_of_trips,total_days,run_rate_trips_per_day,pct_total
0,working_day_morning,4387263,260,16874.0,82.96
1,weekend_morning,901000,104,8663.0,17.04


In [ ]:
# calculate total number of trips as an hour-of-day (0-23) x day-of-week heatmap: groupBy(day_of_week).pivot(hour).count() -- marginal sums give per-hour and per-day totals for free